In [1]:
!pip install x-transformers pythainlp -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import gc
import os

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score
from torch.amp import autocast, GradScaler

from transformers import AutoTokenizer, AutoModel
from x_transformers.x_transformers import AttentionLayers
from pythainlp.tokenize import word_tokenize
from tqdm import tqdm

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ===== CONFIG =====
TRAIN_CSV = "/kaggle/input/prachatai-dataset/prachatai_train.csv"
VAL_CSV   = "/kaggle/input/prachatai-dataset/prachatai_validation.csv"
TEST_CSV  = "/kaggle/input/prachatai-dataset/prachatai_test.csv"

MODEL_PATH = "mbert_xt_pythai.pt"

# ===== RESUME CONFIG =====
EPOCHS = 100           

MAX_LEN = 256
BATCH_SIZE = 128
LR = 2e-4

LABEL_COLS = [
    "politics", "human_rights", "quality_of_life", "international",
    "social", "environment", "economics", "culture", "labor",
    "national_security", "ict", "education"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Resume from epoch {START_EPOCH} to {EPOCHS}")

Device: cuda
Resume from epoch 80 to 100


In [4]:
hf_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
PAD_ID = hf_tokenizer.pad_token_id
print("Tokenizer loaded!")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Tokenizer loaded!


In [5]:
class UltraLazyDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path, usecols=["body_text"] + LABEL_COLS)
        self.n = len(self.df)
        print(f"Loaded {self.n} samples from {csv_path}")
        
    def __len__(self):
        return self.n
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["body_text"])
        label = row[LABEL_COLS].values.astype(np.float32)
        
        try:
            words = word_tokenize(text[:1000], engine="newmm")
        except:
            words = text[:1000].split()
            
        enc = hf_tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=MAX_LEN,
            padding=False
        )
        
        return torch.tensor(enc["input_ids"], dtype=torch.long), torch.tensor(label, dtype=torch.float32)

print("Creating datasets...")
train_dataset = UltraLazyDataset(TRAIN_CSV)
val_dataset = UltraLazyDataset(VAL_CSV)
test_dataset = UltraLazyDataset(TEST_CSV)
gc.collect()

Creating datasets...
Loaded 54379 samples from /kaggle/input/prachatai-dataset/prachatai_train.csv
Loaded 6721 samples from /kaggle/input/prachatai-dataset/prachatai_validation.csv
Loaded 6789 samples from /kaggle/input/prachatai-dataset/prachatai_test.csv


34

In [6]:
def collate_fn(batch):
    seqs, labels = zip(*batch)
    padded = pad_sequence(seqs, batch_first=True, padding_value=PAD_ID)
    attn_mask = (padded != PAD_ID).long()
    return padded.to(device), attn_mask.to(device), torch.stack(labels).to(device)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
print(f"DataLoaders ready! Train batches: {len(train_loader)}")

DataLoaders ready! Train batches: 425


In [7]:
encoder = AutoModel.from_pretrained("bert-base-multilingual-cased")
for p in encoder.parameters():
    p.requires_grad = False
print("Encoder loaded (frozen)")

2026-02-20 15:58:15.849664: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771603096.019678      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771603096.070857      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771603096.497714      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771603096.497755      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771603096.497758      23 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Encoder loaded (frozen)


In [8]:
class MBertXTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = encoder
        self.hidden = 768

        self.decoder = AttentionLayers(
            dim=self.hidden,
            depth=4,
            heads=4,
            cross_attend=True,
            causal=False
        )
        self.classifier = nn.Linear(self.hidden, len(LABEL_COLS))

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            enc = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        
        x = torch.zeros_like(enc)
        dec = self.decoder(x, context=enc, context_mask=attention_mask.bool())
        
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (dec * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.classifier(pooled)

model = MBertXTClassifier().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scaler = GradScaler()

print(f"Model created! Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params")

Model created! Trainable: 25,200,396 params


In [9]:
# ===== LOAD CHECKPOINT =====
if os.path.exists(CHECKPOINT_PATH):
    model.load_state_dict(torch.load(CHECKPOINT_PATH))
    print(f"✅ Loaded checkpoint from: {CHECKPOINT_PATH}")
    print(f"✅ Resuming from epoch {START_EPOCH}")
else:
    print(f"❌ Checkpoint not found at: {CHECKPOINT_PATH}")
    print("Please check the path and try again!")

✅ Loaded checkpoint from: /kaggle/input/datasets/kmkimmy/mbert-checkpoint3/mbert_xt_pythai(1).pt
✅ Resuming from epoch 80


In [10]:
# ===== RESUME TRAINING =====
print(f"\nResuming training from epoch {START_EPOCH} to {EPOCHS}...")
best_f1 = 0.0

for epoch in range(START_EPOCH, EPOCHS):
    model.train()
    total_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for i, (Xb, maskb, yb) in enumerate(pbar):
        optimizer.zero_grad()
        
        with autocast(device_type="cuda"):
            logits = model(Xb, maskb)
            loss = criterion(logits, yb)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        
        if (i + 1) % 50 == 0:
            torch.cuda.empty_cache()
            pbar.set_postfix({"loss": f"{total_loss/(i+1):.4f}"})

    # Validation
    model.eval()
    yt, yp = [], []
    with torch.no_grad():
        for Xb, maskb, yb in val_loader:
            with autocast(device_type="cuda"):
                logits = model(Xb, maskb)
                preds = (torch.sigmoid(logits) > 0.5).int()
            yt.append(yb.cpu().numpy())
            yp.append(preds.cpu().numpy())

    val_f1 = f1_score(np.vstack(yt), np.vstack(yp), average="macro")
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1:03d} | Loss {avg_loss:.4f} | Val F1 {val_f1:.4f}")

    # Save best model
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"  💾 Saved best model (F1={val_f1:.4f})")
    
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n✅ Training complete! Best Val F1: {best_f1:.4f}")


Resuming training from epoch 80 to 100...


Epoch 81: 100%|██████████| 425/425 [16:22<00:00,  2.31s/it, loss=0.1011]


Epoch 081 | Loss 0.1014 | Val F1 0.6013
  💾 Saved best model (F1=0.6013)


Epoch 82: 100%|██████████| 425/425 [16:25<00:00,  2.32s/it, loss=0.0941]


Epoch 082 | Loss 0.0944 | Val F1 0.5968


...

Epoch 098 | Loss 0.0375 | Val F1 0.5847


Epoch 99: 100%|██████████| 425/425 [16:24<00:00,  2.32s/it, loss=0.0363]


Epoch 099 | Loss 0.0364 | Val F1 0.5885


Epoch 100: 100%|██████████| 425/425 [16:22<00:00,  2.31s/it, loss=0.0349]


Epoch 100 | Loss 0.0351 | Val F1 0.5822

✅ Training complete! Best Val F1: 0.6013


In [11]:
# ===== TEST EVALUATION =====
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH))
    print("Loaded best model!")
else:
    print("Using current model")

model.eval()

yt, yp = [], []
with torch.no_grad():
    for Xb, maskb, yb in tqdm(test_loader, desc="Testing"):
        with autocast(device_type="cuda"):
            logits = model(Xb, maskb)
            preds = (torch.sigmoid(logits) > 0.5).int()
        yt.append(yb.cpu().numpy())
        yp.append(preds.cpu().numpy())

test_f1 = f1_score(np.vstack(yt), np.vstack(yp), average="macro")
print(f"\n🎯 Test F1 Score: {test_f1:.4f}")

Loaded best model!


Testing: 100%|██████████| 54/54 [01:31<00:00,  1.69s/it]


🎯 Test F1 Score: 0.5977


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

yt, yp = [], []
with torch.no_grad():
    for Xb, maskb, yb in tqdm(test_loader, desc="Testing"):
        with autocast(device_type="cuda"):
            logits = model(Xb, maskb)
            preds = (torch.sigmoid(logits) > 0.5).int()
        yt.append(yb.cpu().numpy())
        yp.append(preds.cpu().numpy())

yt_arr = np.vstack(yt)
yp_arr = np.vstack(yp)

# F1 Score
test_f1 = f1_score(yt_arr, yp_arr, average="macro")
test_f1_micro = f1_score(yt_arr, yp_arr, average="micro")

# Accuracy
label_acc = (yt_arr == yp_arr).mean()  # per-label accuracy
exact_match = accuracy_score(yt_arr, yp_arr)  # exact match (all labels correct)

print(f"\n{'='*50}")
print(f"  Results after 100 epochs")
print(f"{'='*50}")
print(f"  F1 Score (Macro):   {test_f1:.4f}")
print(f"  F1 Score (Micro):   {test_f1_micro:.4f}")
print(f"  Label Accuracy:     {label_acc:.4f}")
print(f"  Exact Match Acc:    {exact_match:.4f}")
print(f"{'='*50}")

print("\nPer-class Report:")
print(classification_report(yt_arr, yp_arr, target_names=LABEL_COLS, zero_division=0))

In [12]:
# ===== PREDICTION =====
def predict(text, threshold=0.5):
    words = word_tokenize(text[:500], engine="newmm")
    enc = hf_tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    
    with torch.no_grad():
        with autocast(device_type="cuda"):
            logits = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
            probs = torch.sigmoid(logits)[0].cpu().numpy()
    
    results = [(LABEL_COLS[i], float(probs[i])) for i in range(len(LABEL_COLS)) if probs[i] >= threshold]
    return sorted(results, key=lambda x: x[1], reverse=True)

print("\n===== PREDICT EXAMPLES =====")
print(predict("รัฐบาลไทยประกาศนโยบายด้านสิ่งแวดล้อมใหม่"))
print(predict("แรงงานเรียกร้องสิทธิ์การทำงาน"))


===== PREDICT EXAMPLES =====
[('environment', 0.9619140625)]
[('labor', 0.8955078125)]
